In [2]:
!pip install drain3

import os
import pandas as pd
import re
import time
from drain3 import TemplateMiner
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

drive.mount('/content/drive', force_remount=True)
print(os.listdir('/content/drive/MyDrive'))

log_file_path = "/content/drive/MyDrive/HDFS.log"
print("Setup complete - ready to parse!")

Mounted at /content/drive
['Colab Notebooks', 'Programme%20Hanbook%20MDS.pdf', 'Programme%20Hanbook%20MDS.gdoc', 'HDFS.log', 'anomaly_label.csv', 'HDFS_structured_labeled.csv', 'model_comparison_results.csv', 'final_model_comparison.csv', 'HDFS_structured_sample.csv']
Setup complete - ready to parse!


In [3]:
template_miner = TemplateMiner()
parsed_results = []

sample_size = 500000

with open(log_file_path, "r", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        line = line.strip()
        if not line:
            continue
        result = template_miner.add_log_message(line)
        parsed_results.append({
            "raw_log": line,
            "template": result["template_mined"],
            "cluster_id": result["cluster_id"]
        })

df = pd.DataFrame(parsed_results)
df.to_csv("/content/drive/MyDrive/HDFS_structured_sample.csv", index=False)
print(f"Done! Parsed {len(df)} log lines into {df['cluster_id'].nunique()} unique templates.")

Done! Parsed 500000 log lines into 35 unique templates.


In [4]:
sample_size = 500000

parsed_results = []

with open(log_file_path, "r", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        line = line.strip()
        if not line:
            continue
        result = template_miner.add_log_message(line)
        parsed_results.append({
            "raw_log": line,
            "template": result["template_mined"],
            "cluster_id": result["cluster_id"]
        })

df = pd.DataFrame(parsed_results)
df.to_csv("/content/drive/MyDrive/HDFS_structured_sample.csv", index=False)

print(f"Done! Parsed {len(df)} log lines into {df['cluster_id'].nunique()} unique templates.")

Done! Parsed 500000 log lines into 34 unique templates.


In [5]:
labels_df = pd.read_csv("/content/drive/MyDrive/anomaly_label.csv")

def extract_block_id(log_line):
    match = re.search(r'(blk_-?\d+)', log_line)
    return match.group(1) if match else None

df['BlockId'] = df['raw_log'].apply(extract_block_id)

df_labeled = df.merge(labels_df, on='BlockId', how='left')
df_labeled = df_labeled.dropna(subset=['Label'])

df_labeled.to_csv("/content/drive/MyDrive/HDFS_structured_labeled.csv", index=False)
print(f"Saved labeled dataset! Remaining rows: {len(df_labeled)}")
print(df_labeled['Label'].value_counts())

Saved labeled dataset! Remaining rows: 500000
Label
Normal     480905
Anomaly     19095
Name: count, dtype: int64


In [6]:
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(df_labeled['template'])
y = df_labeled['Label'].apply(lambda x: 1 if x == 'Anomaly' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 400000, Test size: 100000


In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": LinearSVC(max_iter=2000),
    "Naive Bayes": MultinomialNB()
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    y_pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "Train Time (s)": train_time
    })
    print(f"{name} done in {train_time:.2f}s.")

results_df = pd.DataFrame(results)
results_df.to_csv("/content/drive/MyDrive/model_comparison_results.csv", index=False)
print(results_df)

Training Logistic Regression...
Logistic Regression done in 1.87s.
Training Random Forest...
Random Forest done in 62.66s.
Training SVM...
SVM done in 9.58s.
Training Naive Bayes...
Naive Bayes done in 0.08s.
                 Model  Accuracy  Precision    Recall  F1-score  \
0  Logistic Regression   0.96432   0.943463  0.069914  0.130180   
1        Random Forest   0.96432   0.943463  0.069914  0.130180   
2                  SVM   0.96432   0.943463  0.069914  0.130180   
3          Naive Bayes   0.96405   0.959016  0.061273  0.115186   

   Train Time (s)  
0        1.871208  
1       62.656462  
2        9.583255  
3        0.076536  


In [8]:
models_balanced = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "SVM": LinearSVC(class_weight='balanced', max_iter=2000),
    "Naive Bayes": MultinomialNB()
}

results_balanced = []
for name, model in models_balanced.items():
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    y_pred = model.predict(X_test)
    results_balanced.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "Train Time (s)": train_time
    })

results_balanced_df = pd.DataFrame(results_balanced)
results_balanced_df.to_csv("/content/drive/MyDrive/model_comparison_results_balanced.csv", index=False)
print(results_balanced_df)

                 Model  Accuracy  Precision    Recall  F1-score  \
0  Logistic Regression   0.67394   0.054036  0.456664  0.096637   
1        Random Forest   0.67394   0.054036  0.456664  0.096637   
2                  SVM   0.67394   0.054036  0.456664  0.096637   
3          Naive Bayes   0.96405   0.959016  0.061273  0.115186   

   Train Time (s)  
0        3.536514  
1       78.410611  
2      156.784587  
3        0.072192  


In [9]:
final_comparison = results_df.merge(results_balanced_df, on="Model", suffixes=("_before", "_after"))
final_comparison = final_comparison[[
    "Model", "Accuracy_before", "Accuracy_after",
    "Precision_before", "Precision_after",
    "Recall_before", "Recall_after",
    "F1-score_before", "F1-score_after"
]]
final_comparison.to_csv("/content/drive/MyDrive/final_model_comparison.csv", index=False)
print(final_comparison)

                 Model  Accuracy_before  Accuracy_after  Precision_before  \
0  Logistic Regression          0.96432         0.67394          0.943463   
1        Random Forest          0.96432         0.67394          0.943463   
2                  SVM          0.96432         0.67394          0.943463   
3          Naive Bayes          0.96405         0.96405          0.959016   

   Precision_after  Recall_before  Recall_after  F1-score_before  \
0         0.054036       0.069914      0.456664         0.130180   
1         0.054036       0.069914      0.456664         0.130180   
2         0.054036       0.069914      0.456664         0.130180   
3         0.959016       0.061273      0.061273         0.115186   

   F1-score_after  
0        0.096637  
1        0.096637  
2        0.096637  
3        0.115186  
